# Mask3D BIM — Treino no Colab Pro (A100)

**Setup completo:** ~25-30 min (ME domina com ~20 min de compilação)

**Pré-requisitos no Google Drive:**
- `mask3d_dataset_bim.tar.gz` (2.12 GB) — dataset com 1289 cenas .npz
- `scannet_val.ckpt` (151 MB) — checkpoint pré-treinado ScanNet
- `train_mask3d.py` — script de treino

**Runtime:** A100 40GB, CUDA 12, PyTorch 2.1.2+cu121, ME 0.5.4

---
⚠️ **A célula 1 reinicia o kernel automaticamente.** Depois do restart, continue da célula 2.

## 1. Instalar CondaColab
⚠️ **Kernel reinicia automaticamente após esta célula.** É normal.

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()  # kernel reinicia aqui

## 2. Criar env conda `me` (Python 3.10)
▶️ **Rode esta célula DEPOIS do restart do kernel.**

In [ ]:
import condacolab
condacolab.check()

import subprocess
# Remove env quebrado se existir (da tentativa anterior sem pip)
subprocess.run(["conda", "env", "remove", "-n", "me", "-y"])
# Cria env com pip incluido
subprocess.run([
    "conda", "create", "-n", "me", "python=3.10",
    "pip", "setuptools=69.5.1", "ninja", "-y"
], check=True)
print("\n✅ Env 'me' criado com Python 3.10 + pip")

## 3. Instalar PyTorch 2.1.2 + cu121

In [ ]:
!/opt/conda/envs/me/bin/pip install torch==2.1.2+cu121 torchvision==0.16.2+cu121 \
  --extra-index-url https://download.pytorch.org/whl/cu121 2>&1 | tail -5
print("\n✅ PyTorch 2.1.2+cu121 instalado")

## 4. Instalar CUDA 12.1 toolkit (nvcc)

In [ ]:
!apt-get install -y cuda-toolkit-12-1 2>&1 | tail -3
!ls /usr/local/cuda-12.1/bin/nvcc && echo "\n✅ nvcc 12.1 OK"

## 5. Patch cpp_extension.py (silenciar erro de versão CUDA)

In [ ]:
!sed -i 's/raise RuntimeError(CUDA_MISMATCH_MESSAGE/warnings.warn(CUDA_MISMATCH_MESSAGE/' \
  /opt/conda/envs/me/lib/python3.10/site-packages/torch/utils/cpp_extension.py
print("✅ Patch cpp_extension.py aplicado")

## 6. Clonar MinkowskiEngine + patches Thrust (issue #601)

In [ ]:
%%bash
cd /content && git clone https://github.com/NVIDIA/MinkowskiEngine.git 2>&1 | tail -2
cd /content/MinkowskiEngine

# Patch 1: concurrent_unordered_map.cuh
sed -i '/#include <thrust\/pair.h>/a #include <thrust\/execution_policy.h>' \
  src/3rdparty/concurrent_unordered_map.cuh

# Patch 2: convolution_kernel.cuh
sed -i '/#include <thrust\/functional.h>/a #include <thrust\/execution_policy.h>' \
  src/convolution_kernel.cuh

# Patch 3: coordinate_map_gpu.cu
sed -i '/#include <thrust\/sort.h>/a #include <thrust\/unique.h>\n#include <thrust\/remove.h>' \
  src/coordinate_map_gpu.cu

# Patch 4: spmm.cu (anchor eh <cusparse.h>, NAO <thrust/device_vector.h>)
sed -i '/#include <cusparse.h>/a #include <thrust\/device_vector.h>\n#include <thrust\/execution_policy.h>\n#include <thrust\/sort.h>\n#include <thrust\/reduce.h>' \
  src/spmm.cu

# Verifica patch 4
echo "--- spmm.cu linhas 30-35 ---"
sed -n '30,35p' src/spmm.cu
echo ""
echo "✅ 4 patches Thrust aplicados"

## 7. Compilar MinkowskiEngine (~20 min no A100)
☕ Vá tomar um café. Se o terminal desconectar, `tmux attach` reconecta.

In [ ]:
%%bash
cd /content/MinkowskiEngine && \
CUDA_HOME=/usr/local/cuda-12.1 TORCH_CUDA_ARCH_LIST="8.0" MAX_JOBS=4 FORCE_CUDA=1 \
/opt/conda/envs/me/bin/python setup.py install \
--blas_include_dirs=/usr/include --blas=openblas --force_cuda 2>&1 | tail -20
echo "EXIT CODE: $?"

## 8. Verificar MinkowskiEngine

In [ ]:
!/opt/conda/envs/me/bin/python -c "import MinkowskiEngine as ME; print('✅ ME version:', ME.__version__)"

## 9. Instalar dependências Python

In [ ]:
!/opt/conda/envs/me/bin/pip install -q \
  open3d omegaconf scipy hydra-core \
  pytorch-lightning==1.9.5 torchmetrics==0.11.4 wandb
print("✅ Dependências instaladas")

## 10. Clonar Mask3D + instalar PointNet2

In [ ]:
%%bash
cd /content && git clone https://github.com/JonasSchult/Mask3D.git 2>&1 | tail -2

cd /content/Mask3D/third_party/pointnet2 && \
CUDA_HOME=/usr/local/cuda-12.1 TORCH_CUDA_ARCH_LIST="8.0" \
/opt/conda/envs/me/bin/python setup.py install 2>&1 | tail -5
echo "✅ PointNet2 instalado"

---
## 11. Montar Google Drive
⚠️ **Tem que rodar como célula do notebook** (não funciona do terminal). Popup OAuth pode bloquear na 1ª vez — tente de novo.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Drive montado")

## 12. Extrair dataset + copiar script
Ajuste os caminhos se seus arquivos estiverem em outra pasta do Drive.

In [ ]:
import os, glob

DRIVE = "/content/drive/MyDrive"

# Extrair dataset
TARBALL = f"{DRIVE}/mask3d_dataset_bim.tar.gz"
DATA_DIR = "/content/mask3d_data"

if not os.path.exists(DATA_DIR):
    os.makedirs(DATA_DIR, exist_ok=True)
    !tar -xzf "{TARBALL}" -C "{DATA_DIR}"
    print(f"Extraído para {DATA_DIR}")
else:
    print(f"{DATA_DIR} já existe, pulando extração")

# Encontrar onde os .npz ficaram (pode ter subpasta)
npz_files = glob.glob(f"{DATA_DIR}/**/*.npz", recursive=True)
if not npz_files:
    npz_files = glob.glob(f"{DATA_DIR}/*.npz")

# Se os .npz estão numa subpasta, apontar pra ela
if npz_files:
    real_data_dir = os.path.dirname(npz_files[0])
    print(f"\n✅ {len(npz_files)} cenas .npz encontradas em: {real_data_dir}")
else:
    print("❌ Nenhum .npz encontrado! Verifique o tar.gz")
    real_data_dir = DATA_DIR

# Copiar script de treino
SCRIPT_SRC = f"{DRIVE}/train_mask3d.py"
SCRIPT_DST = "/content/train_mask3d.py"
if os.path.exists(SCRIPT_SRC):
    !cp "{SCRIPT_SRC}" "{SCRIPT_DST}"
    print(f"✅ train_mask3d.py copiado para {SCRIPT_DST}")
else:
    print(f"⚠️ {SCRIPT_SRC} não encontrado no Drive — faça upload manual")
    # Upload interativo como fallback
    from google.colab import files
    print("Faça upload do train_mask3d.py:")
    uploaded = files.upload()
    if uploaded:
        !mv train_mask3d.py /content/
        print("✅ Upload feito")

## 13. Verificação final antes do treino

In [ ]:
%%bash
echo "=== GPU ==="
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
echo ""
echo "=== MinkowskiEngine ==="
/opt/conda/envs/me/bin/python -c "import MinkowskiEngine as ME; print(f'ME {ME.__version__}')"
echo ""
echo "=== Dataset ==="
ls /content/mask3d_data/**/*.npz 2>/dev/null | wc -l | xargs -I{} echo "{} cenas .npz"
ls /content/mask3d_data/*.npz 2>/dev/null | wc -l | xargs -I{} echo "(raiz: {} cenas)"
echo ""
echo "=== Checkpoint ==="
ls -lh /content/drive/MyDrive/scannet_val.ckpt 2>/dev/null || echo "❌ scannet_val.ckpt não encontrado"
echo ""
echo "=== Script ==="
ls -lh /content/train_mask3d.py 2>/dev/null || echo "❌ train_mask3d.py não encontrado"

---
## 14. 🚀 TREINO

Configuração:
- **LR:** 1e-4 (decoder) / 1e-5 (backbone) — diferenciado automaticamente
- **Backbone:** unfrozen (39.6M params total)
- **max_voxels:** 80000 (A100 aguenta)
- **Checkpoints:** salvos no Drive (pasta `treino/`)

Ajuste `--data` abaixo se os .npz ficaram numa subpasta.

In [ ]:
!/opt/conda/envs/me/bin/python /content/train_mask3d.py \
    --data {real_data_dir} \
    --ckpt /content/drive/MyDrive/scannet_val.ckpt \
    --ckpt_out /content/drive/MyDrive/treino \
    --epochs 50 \
    --lr 1e-4 \
    --max_voxels 80000

---
## Resumir treino (se sessão caiu)
Se a sessão morrer no meio, refaça as células 1-11, depois rode esta com `--start_epoch`:

In [ ]:
# Descomente e ajuste o epoch:
# !/opt/conda/envs/me/bin/python /content/train_mask3d.py \
#     --data /content/mask3d_data \
#     --ckpt /content/drive/MyDrive/treino/best_bim.ckpt \
#     --ckpt_out /content/drive/MyDrive/treino \
#     --epochs 50 \
#     --start_epoch 15 \
#     --lr 1e-4 \
#     --max_voxels 80000